In [1]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import pickle
import pandas as pd
from tqdm.notebook import tqdm
from scipy.special import ndtri
from scipy.stats import norm
from scipy.stats import bootstrap
from sklearn.metrics import r2_score

from methyldl.deconvolution.evaluation import compute_deconvolution_metrics

%load_ext autoreload
%autoreload 2

In [2]:
ROOT_DATA_DIR = Path(
    "/staging/leuven/stg_00118/methylDL/experiments/ExtendedProportions/pseudobulk/methylBert_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_postfiltered_min_length_50_soft_labels_pooled_jakkard_no_data_leak_d041"
)
CALIBRATION_RESULTS_DIR = (
    ROOT_DATA_DIR
    / "pseudobulk/fitted_deconvolvers_unifrorm_multinomial_all_top156features/callibration"
)

## Data loading

In [3]:
# Load test predictions for all deconvolvers and calibration methods
DECONVOLVER_NAMES = ["psls"]  # ["nnls", "psls", "xgb", "swn", "mlp"]
CALIBRATION_METHODS = [
    "uncalibrated",
    # "linear_clip01_normalize",
    "linear_clip0_normalize",
    "linear_simplex_projection",
    "vector_scaling",
]

predictions = {}  # {deconv_name: {method: test_pred array}}
target_proportions = None

for deconv_name in DECONVOLVER_NAMES:
    deconv_dir = CALIBRATION_RESULTS_DIR / f"{deconv_name}_calibrators_and_predictions"
    if not deconv_dir.exists():
        print(f"WARNING: {deconv_dir} not found, skipping")
        continue

    predictions[deconv_name] = {}

    # Uncalibrated predictions (also contains targets)
    uncalib_path = deconv_dir / "uncalibrated_predictions.npz"
    if uncalib_path.exists():
        data = np.load(uncalib_path)
        predictions[deconv_name]["uncalibrated"] = data["test_pred"]
        # Load target proportions (same across all deconvolvers)
        if target_proportions is None:
            target_proportions = data["test_target"]

    # Linear calibrated predictions
    for norm_method in ["clip0_normalize", "simplex_projection"]:  # "clip01_normalize",
        pred_path = deconv_dir / f"linear_{norm_method}_predictions.npz"
        if pred_path.exists():
            data = np.load(pred_path)
            predictions[deconv_name][f"linear_{norm_method}"] = data["test_pred"]

    # Vector scaling predictions
    vs_path = deconv_dir / "vector_scaling_predictions.npz"
    if vs_path.exists():
        data = np.load(vs_path)
        predictions[deconv_name]["vector_scaling"] = data["test_pred"]

print(f"Loaded predictions for deconvolvers: {list(predictions.keys())}")
for name, methods in predictions.items():
    print(f"  {name}: {list(methods.keys())}")
print(f"Target proportions shape: {target_proportions.shape}")

Loaded predictions for deconvolvers: ['psls']
  psls: ['uncalibrated', 'linear_clip0_normalize', 'linear_simplex_projection', 'vector_scaling']
Target proportions shape: (100000, 39)


## Functions

In [ ]:
def bca_ci_mean(data, n_resamples=10_000, confidence_level=0.95, random_state=42):
    """BCa bootstrap CI for np.mean on the full dataset.

    The jackknife is computed analytically for the mean:
        jack_i = (sum(data) - data[i]) / (n - 1)
    """
    rng = np.random.default_rng(random_state)
    n = len(data)
    theta_hat = np.mean(data)

    # Bootstrap distribution (full n-out-of-n resamples)
    boot_stats = np.empty(n_resamples)
    for i in range(n_resamples):
        boot_stats[i] = np.mean(data[rng.integers(0, n, size=n)])

    # Bias correction (z0)
    z0 = ndtri(np.mean(boot_stats < theta_hat))

    # Acceleration (a) — analytical jackknife for the mean
    total = np.sum(data)
    jack_stats = (total - data) / (n - 1)  # 1-D array, O(n) memory
    jack_mean = np.mean(jack_stats)
    diff = jack_mean - jack_stats
    a = np.sum(diff**3) / (6.0 * np.sum(diff**2) ** 1.5)

    # Adjusted percentiles
    alpha = (1 - confidence_level) / 2
    z_lo, z_hi = ndtri(alpha), ndtri(1 - alpha)
    q_lo = norm.cdf(z0 + (z0 + z_lo) / (1 - a * (z0 + z_lo)))
    q_hi = norm.cdf(z0 + (z0 + z_hi) / (1 - a * (z0 + z_hi)))

    return np.percentile(boot_stats, [q_lo * 100, q_hi * 100])

## MSE CI

In [ ]:
# Bootstrap 95% BCa CI for MSE — PSLS only, full dataset
psls_preds = predictions["psls"]

results_bca_mse = {}
for method, test_pred in tqdm(psls_preds.items()):
    pointwise_mse = np.mean((test_pred - target_proportions) ** 2, axis=1)
    mse_point = np.mean(pointwise_mse)

    ci_lower, ci_upper = bca_ci_mean(pointwise_mse, n_resamples=10_000)

    results_bca_mse[method] = {
        "MSE": mse_point,
        "CI_lower": ci_lower,
        "CI_upper": ci_upper,
    }

results_bca_mse_df = pd.DataFrame(results_bca_mse).T
results_bca_mse_df.index.name = "calibration_method"
(results_bca_mse_df * 1e4).round(2)

## MAE CI

In [ ]:
# Bootstrap 95% BCa CI for MAE — PSLS only, full dataset
psls_preds = predictions["psls"]

results_bca_mae = {}
for method, test_pred in tqdm(psls_preds.items()):
    pointwise_mae = np.mean(np.abs(test_pred - target_proportions), axis=1)
    mae_point = np.mean(pointwise_mae)

    ci_lower, ci_upper = bca_ci_mean(pointwise_mae, n_resamples=10_000)

    results_bca_mae[method] = {
        "MAE": mae_point,
        "CI_lower": ci_lower,
        "CI_upper": ci_upper,
    }

results_bca_mae_df = pd.DataFrame(results_bca_mae).T
results_bca_mae_df.index.name = "calibration_method"
(results_bca_mae_df * 1e3).round(2)

## KL CI

In [ ]:
# Bootstrap 95% BCa CI for KL — PSLS only, full dataset
psls_preds = predictions["psls"]

results_bca_kl = {}
eps = 1e-8
for method, test_pred in tqdm(psls_preds.items()):
    # formula : (target * (target.clamp(min=eps).log() - pred.clamp(min=eps).log()))
    pointwise_kl = np.sum(
        target_proportions
        * (
            np.log(np.clip(target_proportions, eps, None))
            - np.log(np.clip(test_pred, eps, None))
        ),
        axis=1,
    )
    kl_point = np.mean(pointwise_kl)

    ci_lower, ci_upper = bca_ci_mean(pointwise_kl, n_resamples=10_000)

    results_bca_kl[method] = {
        "KL": kl_point,
        "CI_lower": ci_lower,
        "CI_upper": ci_upper,
    }

results_bca_kl_df = pd.DataFrame(results_bca_kl).T
results_bca_kl_df.index.name = "calibration_method"
(results_bca_kl_df * 1e2).round(2)

## $R^2$ CI

In [ ]:
def r2_variance_weighted(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean(axis=0)) ** 2)
    return 1.0 - ss_res / ss_tot


def bca_ci_r2(
    preds, targets, n_resamples=10_000, confidence_level=0.95, random_state=42
):
    """BCa bootstrap CI for R2 (variance weighted) on the full dataset.

    The jackknife is computed analytically for r2.

    Args:
        preds: (n_samples, n_celltypes) array of predicted proportions
        targets: (n_samples, n_celltypes) array of true proportions
    """
    rng = np.random.default_rng(random_state)
    n = len(preds)

    # pre compute per-sample ss_res and ss_tot for the full dataset
    target_avg_over_ctypes = targets.mean(axis=0)  # (n_ctypes, )
    ss_tot_per_sample = np.sum(
        (targets - target_avg_over_ctypes) ** 2, axis=1
    )  # (n_samples,)
    ss_res_per_sample = np.sum((targets - preds) ** 2, axis=1)  # (n_samples,)
    ss_res_sum = np.sum(ss_res_per_sample)
    ss_tot_sum = np.sum(ss_tot_per_sample)

    # Bootstrap distribution (full n-out-of-n resamples)
    boot_stats = np.empty(n_resamples)
    for i in range(n_resamples):
        indices = rng.integers(0, n, size=n)
        boot_stats[i] = 1 - np.sum(ss_res_per_sample[indices]) / np.sum(
            ss_tot_per_sample[indices]
        )

    # Bias correction (z0)
    r2_hat = 1 - ss_res_sum / ss_tot_sum
    z0 = ndtri(np.mean(boot_stats < r2_hat))

    # Acceleration (a) — analytical jackknife for r2

    jack_stats = 1 - (ss_res_sum - ss_res_per_sample) / (
        ss_tot_sum - ss_tot_per_sample
    )  # 1-D array, O(n) memory
    jack_mean = np.mean(jack_stats)
    diff = jack_mean - jack_stats
    a = np.sum(diff**3) / (6.0 * np.sum(diff**2) ** 1.5)

    # Adjusted percentiles
    alpha = (1 - confidence_level) / 2
    z_lo, z_hi = ndtri(alpha), ndtri(1 - alpha)
    q_lo = norm.cdf(z0 + (z0 + z_lo) / (1 - a * (z0 + z_lo)))
    q_hi = norm.cdf(z0 + (z0 + z_hi) / (1 - a * (z0 + z_hi)))

    ci_lower, ci_upper = np.percentile(boot_stats, [q_lo * 100, q_hi * 100])
    return r2_hat, ci_lower, ci_upper

In [ ]:
# Bootstrap 95% CI for R2 — PSLS only, full dataset
psls_preds = predictions["psls"]

results_bca_r2 = {}
eps = 1e-8
for method, test_pred in tqdm(psls_preds.items()):

    r2_point, ci_lower, ci_upper = bca_ci_r2(test_pred, target_proportions)

    results_bca_r2[method] = {
        "R2": r2_point,
        "CI_lower": ci_lower,
        "CI_upper": ci_upper,
    }

results_bca_r2_df = pd.DataFrame(results_bca_r2).T
results_bca_r2_df.index.name = "calibration_method"
(results_bca_r2_df * 1e2).round(2)

## Test wrapper function

In [8]:
methods_results = {}
psls_preds = predictions["psls"]
for method, test_pred in tqdm(psls_preds.items()):
    methods_results[method] = compute_deconvolution_metrics(
        test_pred, target_proportions, compute_ci=True
    )

  0%|          | 0/4 [00:00<?, ?it/s]

In [10]:
methods_results["uncalibrated"]

{'mse': np.float64(0.00017668990324348557),
 'mse_ci_lower': np.float64(0.00017412058518074738),
 'mse_ci_upper': np.float64(0.00017928519176576985),
 'mae': np.float64(0.0036298558228046156),
 'mae_ci_lower': np.float64(0.0036133636352552907),
 'mae_ci_upper': np.float64(0.003646204741690667),
 'kl': np.float64(0.09256406034017414),
 'kl_ci_lower': np.float64(0.09165523790482916),
 'kl_ci_upper': np.float64(0.09352691921082291),
 'r2': np.float64(0.9787112468134751),
 'r2_ci_lower': np.float64(0.9784046743139099),
 'r2_ci_upper': np.float64(0.979015917972715),
 'max_error': 0.3994392398931723,
 'cosine_sim': 0.9905224319194228,
 'loa_lower': -0.026053255220090138,
 'loa_upper': 0.026053255173297624,
 'loa_width': 0.05210651039338776,
 'worst_class_idx': 11,
 'worst_class_name': 11,
 'worst_class_loa_lower': -0.08784279403150978,
 'worst_class_loa_upper': 0.07845340791144058,
 'worst_class_loa_width': 0.16629620194295036,
 'per_class_loa': {'bias': array([-1.99633952e-03,  2.37602299e-

In [14]:
# print results in latex table format
metric_scale_factors = {
    "mae": 1e3,
    "mse": 1e4,
    "kl": 1e2,
    "r2": 1e2,
    "loa": 1e2,
}
method_labels = {
    "uncalibrated": "None",
    "linear_clip0_normalize": "Lin. clip+norm",
    "linear_simplex_projection": "Lin. simplex",
    "vector_scaling": "Vec. scaling",
}

for method_name, method_label in method_labels.items():
    if method_name not in methods_results:
        print(f"WARNING: {method_name} not found in results, skipping")
        continue

    metrics = methods_results[method_name]
    print(
        f"&&{method_label} & "
        f"{(metrics["r2"]*metric_scale_factors["r2"]):.2f} {{\\scriptsize[{(metrics["r2_ci_lower"]*metric_scale_factors["r2"]):.2f}, {(metrics["r2_ci_upper"]*metric_scale_factors["r2"]):.2f}]}} & "
        f"[{metrics["loa_lower"]*metric_scale_factors["loa"]:.2f}, {(metrics["loa_upper"]*metric_scale_factors["loa"]):.2f}] & "
        f"[{metrics["worst_class_loa_lower"]*metric_scale_factors["loa"]:.2f}, {(metrics["worst_class_loa_upper"]*metric_scale_factors["loa"]):.2f}] & "
        f"{(metrics["mae"]*metric_scale_factors["mae"]):.2f} {{\\scriptsize[{(metrics["mae_ci_lower"]*metric_scale_factors["mae"]):.2f}, {(metrics["mae_ci_upper"]*metric_scale_factors["mae"]):.2f}]}} & "
        f"{(metrics["mse"]*metric_scale_factors["mse"]):.2f} {{\\scriptsize[{(metrics["mse_ci_lower"]*metric_scale_factors["mse"]):.2f}, {(metrics["mse_ci_upper"]*metric_scale_factors["mse"]):.2f}]}} & "
        f"{(metrics["kl"]*metric_scale_factors["kl"]):.2f} {{\\scriptsize[{(metrics["kl_ci_lower"]*metric_scale_factors["kl"]):.2f}, {(metrics["kl_ci_upper"]*metric_scale_factors["kl"]):.2f}]}} \\\\"
    )

&&None & 97.87 {\scriptsize[97.84, 97.90]} & [-2.61, 2.61] & [-8.78, 7.85] & 3.63 {\scriptsize[3.61, 3.65]} & 1.77 {\scriptsize[1.74, 1.79]} & 9.26 {\scriptsize[9.17, 9.35]} \\
&&Lin. clip+norm & 98.19 {\scriptsize[98.17, 98.22]} & [-2.40, 2.40] & [-6.75, 7.98] & 3.47 {\scriptsize[3.45, 3.48]} & 1.50 {\scriptsize[1.48, 1.52]} & 7.54 {\scriptsize[7.48, 7.60]} \\
&&Lin. simplex & 98.51 {\scriptsize[98.49, 98.53]} & [-2.18, 2.18] & [-6.01, 7.52] & 3.16 {\scriptsize[3.15, 3.17]} & 1.24 {\scriptsize[1.22, 1.25]} & 8.34 {\scriptsize[8.23, 8.45]} \\
&&Vec. scaling & 97.66 {\scriptsize[97.64, 97.69]} & [-2.73, 2.73] & [-7.94, 9.89] & 4.06 {\scriptsize[4.05, 4.08]} & 1.94 {\scriptsize[1.92, 1.96]} & 7.69 {\scriptsize[7.61, 7.76]} \\
